In [18]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import mlflow

from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier, Perceptron
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

import joblib

import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)


In [2]:
df = pd.read_csv("../data/processed data/merged_data_V2.csv")
df.head()

,SWEAT index,K index,Totals totals index,Date,TH,Environmental Stability,Moisture Indices,Convective Potential,Temperature Pressure,Moisture Temperature Profiles
0,91.2,-1.4,24.7,1981-01-01,0,25.8,22.8,0.0,5636,993.98
1,75.7,1.6,30.3,1981-01-02,0,21.5,20.4,0.0,5592,956.13
2,64.0,2.8,37.5,1981-01-03,0,1.9,19.7,-259.7,5636,862.29
3,128.3,17.5,41.6,1981-01-04,0,13.8,23.8,0.0,5581,978.71
4,194.2,23.5,50.6,1981-01-05,0,4.0,28.6,-55.4,5578,965.77


## Step 1: Apply transformations
- Log transform (np.log1p): applied to right-skewed features  
- Reflect + log: applied to left-skewed features (negate, shift, then log1p)


In [3]:
log_cols       = ['SWEAT index']
reflect_cols   = ['K index', 'Moisture Indices']
shift_log_cols = ['Convective Potential']

# removing extreme tail
df['Convective Potential'] = df['Convective Potential'].clip(
    lower=df['Convective Potential'].quantile(0.01),
    upper=df['Convective Potential'].quantile(0.99)
)

for col in log_cols:
    df[col] = np.log1p(df[col])

for col in reflect_cols:
    df[col] = np.log1p(df[col].max() - df[col])

for col in shift_log_cols:
    shift = abs(df[col].min()) + 1
    df[col] = np.log1p(df[col] + shift)

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SWEAT index,11873.0,5.059036,0.604400,1.916923,4.657763,5.225209,5.521861,6.747234
K index,11873.0,3.614582,0.382736,0.000000,3.310543,3.558201,3.947390,4.831509
Totals totals index,11873.0,39.922311,9.308756,-17.900000,35.400000,41.700000,45.800000,69.800000
TH,11873.0,0.197591,0.398199,0.000000,0.000000,0.000000,0.000000,1.000000
Environmental Stability,11873.0,7.400615,11.181217,-31.200000,0.000000,5.200000,13.300000,58.000000
Moisture Indices,11873.0,4.034062,0.338040,0.000000,3.799974,4.117410,4.312141,4.822698
Convective Potential,11873.0,6.506657,1.169499,0.693147,5.839804,5.883356,7.516113,8.637750
Temperature Pressure,11873.0,5749.557736,70.223398,5530.000000,5698.000000,5756.000000,5805.000000,5961.000000
Moisture Temperature Profiles,11873.0,951.183852,40.405018,497.250000,935.810000,960.430000,979.840000,1168.110000


## Step 2: Clip Outlier

- Not Removing the Outliers because it's not a data error, they are just extreme value.

In [5]:
def check_outlier(cols) -> pd.DataFrame:
    result = []
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers_low = (df[col] < lower).sum()
        outliers_high = (df[col] > upper).sum()

        total_pct = ((outliers_low + outliers_high) / len(df)) * 100

        if total_pct > 1:
            if outliers_low > 0 and outliers_high > 0:
                decision = "Clip Both Side"
            elif outliers_low > 0:
                decision = "Clip Left Side"
            else:
                decision = "Clip Right Side"
        else:
            decision = "Ignore"

        result.append({
            "Field": col,
            "Lower Side Outlier": outliers_low,
            "Upper Side Outlier": outliers_high,
            "Total Percentage": round(total_pct, 2),
            "Decision": decision
        })

    return pd.DataFrame(result)

In [6]:
check_outlier(df.columns.drop(['TH','Date']))

,Field,Lower Side Outlier,Upper Side Outlier,Total Percentage,Decision
0,SWEAT index,87,0,0.73,Ignore
1,K index,3,0,0.03,Ignore
2,Totals totals index,423,31,3.82,Clip Both Side
3,Environmental Stability,17,323,2.86,Clip Both Side
4,Moisture Indices,75,0,0.63,Ignore
5,Convective Potential,144,0,1.21,Clip Left Side
6,Temperature Pressure,4,0,0.03,Ignore
7,Moisture Temperature Profiles,488,6,4.16,Clip Both Side


In [7]:
def clip_outlier(cols_to_clip, cols) -> pd.DataFrame:
    for col, side in cols_to_clip.items():
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        if side == "both":
            df[col] = df[col].clip(lower=lower, upper=upper)
        elif side == "left":
            df[col] = df[col].clip(lower=lower)
        elif side == "right":
            df[col] = df[col].clip(upper=upper)

    return check_outlier(cols)

In [8]:
cols_to_clip = {
    'Totals totals index': 'both',
    'Environmental Stability': 'both',
    'Convective Potential': 'left',
    'Moisture Temperature Profiles':'both',
}
clip_outlier(cols_to_clip, df.columns.drop(['TH','Date']))

,Field,Lower Side Outlier,Upper Side Outlier,Total Percentage,Decision
0,SWEAT index,87,0,0.73,Ignore
1,K index,3,0,0.03,Ignore
2,Totals totals index,0,0,0.00,Ignore
3,Environmental Stability,0,0,0.00,Ignore
4,Moisture Indices,75,0,0.63,Ignore
5,Convective Potential,0,0,0.00,Ignore
6,Temperature Pressure,4,0,0.03,Ignore
7,Moisture Temperature Profiles,0,0,0.00,Ignore


In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SWEAT index,11873.0,5.059036,0.604400,1.916923,4.657763,5.225209,5.521861,6.747234
K index,11873.0,3.614582,0.382736,0.000000,3.310543,3.558201,3.947390,4.831509
Totals totals index,11873.0,40.115000,8.727950,19.800000,35.400000,41.700000,45.800000,61.400000
TH,11873.0,0.197591,0.398199,0.000000,0.000000,0.000000,0.000000,1.000000
Environmental Stability,11873.0,7.280687,10.827969,-19.950000,0.000000,5.200000,13.300000,33.250000
Moisture Indices,11873.0,4.034062,0.338040,0.000000,3.799974,4.117410,4.312141,4.822698
Convective Potential,11873.0,6.535383,1.054010,3.325340,5.839804,5.883356,7.516113,8.637750
Temperature Pressure,11873.0,5749.557736,70.223398,5530.000000,5698.000000,5756.000000,5805.000000,5961.000000
Moisture Temperature Profiles,11873.0,953.028152,33.141884,869.765000,935.810000,960.430000,979.840000,1045.885000


## Step 3: Feature Scaling and Modeling

In [10]:
# feature selection
X = df[df.columns.drop(['TH', 'Date'])]
y = df['TH']

In [21]:
models = {
    #Linear
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Ridge Classifier": RidgeClassifier(random_state=42),
    "SGD Classifier": SGDClassifier(max_iter=1000, random_state=42),
    "Perceptron": Perceptron(max_iter=1000, random_state=42),
    #Tree
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Extra Tree": ExtraTreeClassifier(random_state=42),
    #Ensemble
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    #Neighbors
    "KNN": KNeighborsClassifier(),
    #SVM
    "SVC": SVC(probability=True, random_state=42),
    "Linear SVC": LinearSVC(random_state=42),
    "Nu SVC": NuSVC(nu=0.1, probability=True, random_state=42),
    #Naive Bayes
    "Gaussian NB": GaussianNB(),
    "Bernoulli NB": BernoulliNB(),
    #Discriminant Analysis
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    #Neural Net
    "MLP": MLPClassifier(max_iter=500, random_state=42),
    #Boosts
    "XGBoost":   XGBClassifier(random_state=42, eval_metric="logloss", verbosity=0),
    "LightGBM":  LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost":  CatBoostClassifier(random_state=42, verbose=0),
    #Baseline
    "Dummy (majority)": DummyClassifier(strategy="most_frequent", random_state=42),
    "Dummy (stratified)": DummyClassifier(strategy="stratified", random_state=42)
}

In [22]:
#train - 80% and test - 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [23]:
no_proba_models = ["Ridge Classifier", "Linear SVC", "Perceptron", "SGD Classifier"]

In [26]:
pipelines_v1 = {}

for name, model in models.items():
    pipelines_v1[name] = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('model',  model)
    ])

results = []

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Thunderstorm Forecasting")

for name, pipeline in pipelines_v1.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                roc_auc = roc_auc_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("scaler", "StandardScaler")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("roc_auc", roc_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "ROC AUC": round(roc_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "ROC AUC": "",
                "Error": e
            })
results_df = pd.DataFrame(results)
results_df["ROC AUC"] = pd.to_numeric(results_df["ROC AUC"], errors='coerce')
results_df = results_df.sort_values("ROC AUC", ascending=False)
results_df

,Model,Accuracy,F1,Precision,Recall,ROC AUC,Error
20,MLP,0.7966,0.0082,0.1111,0.0043,0.7696,
17,Bernoulli NB,0.7036,0.4634,0.3606,0.6482,0.7686,
8,Gradient Boosting,0.8021,0.0637,0.4848,0.0341,0.7674,
18,LDA,0.7920,0.0985,0.3418,0.0576,0.7668,
9,Hist Gradient Boosting,0.7949,0.1411,0.4082,0.0853,0.7664,
23,CatBoost,0.7958,0.1594,0.4259,0.0981,0.7653,
16,Gaussian NB,0.6716,0.4701,0.3450,0.7377,0.7610,
0,Logistic Regression,0.7966,0.1039,0.4000,0.0597,0.7606,
22,LightGBM,0.7937,0.1610,0.4087,0.1002,0.7600,
10,AdaBoost,0.8000,0.0326,0.3636,0.0171,0.7564,
